In [ ]:
# @title 1) Student Info & Config
STUDENT_NAME = "Воропаев Фёдор Игоревич"  # @param {type:"string"}
GROUP = "11-301"         # @param {type:"string"}
ASSIGMENT = "HW6_Debugging_Training"
SEED = 42
START_DATE = "2026-04-21"
DUE_DATE = "2026-04-28"


# HW6 — Отладка обучения и эксперименты (CV)

**Тема:** диагностика проблем обучения, контроль экспериментов, overfit/underfit, LR, дисбаланс, утечки, режимы train/eval.

В этой домашке вы сделаете набор практических функций и маленьких экспериментов на **синтетическом CV-датасете**, чтобы:
- научиться быстро ловить типовые ошибки
- выбирать LR по короткому sweep
- видеть, почему accuracy врёт при дисбалансе
- демонстрировать leakage и ошибку `model.eval()`
- делать ablation по одной правке

Баллы: **100** (тесты ниже каждой задачи).


In [ ]:
# Install deps (Colab)
!pip -q install -U torch torchvision torchaudio scikit-learn matplotlib


In [ ]:
import random
from datetime import datetime, timezone
from typing import Dict, Tuple, List, Optional

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

plt.rcParams["figure.dpi"] = 140

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---------------- Reproducibility ----------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ---------------- Dates for last cell ----------------
def _parse_date(s: str):
    try:
        return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except Exception:
        return None

def _sec(td) -> float:
    return float(td.total_seconds())

start_dt = _parse_date(START_DATE)
due_dt = _parse_date(DUE_DATE)
submission_dt = datetime.now(timezone.utc)

# ---------------- Scoring ----------------
SCORES: Dict[str, float] = {}

def _set_score(task: str, pts: float):
    SCORES[task] = float(pts)

def _total_score() -> float:
    return float(sum(SCORES.values()))

def _print_scores():
    print("Scores:")
    for k in sorted(SCORES.keys()):
        print(f"  {k}: {SCORES[k]:.1f}")
    print("TOTAL:", _total_score())

def _assert(cond: bool, msg: str):
    if not cond:
        raise AssertionError(msg)


## Данные: синтетический датасет «круг vs квадрат»

- класс 0: круг  
- класс 1: квадрат  

Параметры датасета:
- `p_pos` — доля класса 1 (для дисбаланса)
- `label_noise` — вероятность переворота метки (шум разметки)
- `domain_shift` — изменение условий (фон/контраст), имитация сдвига домена


In [ ]:
def draw_sample(label: int, img_size=64, bg=30, fg=210, rng=None):
    if rng is None:
        rng = np.random.RandomState(0)
    img = np.full((img_size, img_size), bg, dtype=np.uint8)

    cx = int(rng.uniform(img_size*0.3, img_size*0.7))
    cy = int(rng.uniform(img_size*0.3, img_size*0.7))
    s  = int(rng.uniform(img_size*0.12, img_size*0.22))

    if label == 0:
        yy, xx = np.ogrid[:img_size, :img_size]
        mask = (xx - cx)**2 + (yy - cy)**2 <= s*s
        img[mask] = fg
    else:
        x1, x2 = max(0, cx-s), min(img_size, cx+s)
        y1, y2 = max(0, cy-s), min(img_size, cy+s)
        img[y1:y2, x1:x2] = fg

    img3 = np.stack([img, img, img], axis=0)  # CHW
    return img3

class ShapesDS(Dataset):
    def __init__(self, n: int, p_pos: float = 0.5, label_noise: float = 0.0,
                 domain_shift: float = 0.0, seed: int = 42):
        self.n = n
        self.p_pos = p_pos
        self.label_noise = label_noise
        self.domain_shift = domain_shift
        self.rng = np.random.RandomState(seed)
        self.labels = (self.rng.rand(n) < p_pos).astype(int)

    def __len__(self): return self.n

    def __getitem__(self, idx):
        y = int(self.labels[idx])
        if self.label_noise > 0 and self.rng.rand() < self.label_noise:
            y = 1 - y

        bg = int(np.clip(30 + self.domain_shift*50 + self.rng.normal(0, 6), 0, 255))
        fg = int(np.clip(210 - self.domain_shift*40 + self.rng.normal(0, 6), 0, 255))

        x = draw_sample(y, rng=self.rng, bg=bg, fg=fg)
        x = torch.from_numpy(x).float() / 255.0
        return x, torch.tensor(y, dtype=torch.long)

def make_loaders(n_train=800, n_dev=400, p_pos=0.5, label_noise=0.0, domain_shift_dev=0.0, seed=123, batch=64):
    train_ds = ShapesDS(n_train, p_pos=p_pos, label_noise=label_noise, domain_shift=0.0, seed=seed)
    dev_ds   = ShapesDS(n_dev,   p_pos=p_pos, label_noise=0.0,        domain_shift=domain_shift_dev, seed=seed+1)
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=2)
    dev_loader   = DataLoader(dev_ds,   batch_size=batch, shuffle=False, num_workers=2)
    return train_loader, dev_loader

# quick sanity sample
tl, dl = make_loaders()
x, y = next(iter(tl))
print("batch x:", tuple(x.shape), "min/max:", float(x.min()), float(x.max()), "y:", y[:10].tolist())


## Модель: маленькая CNN

Она достаточно выразительна для этой задачи, но быстро обучается.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, with_dropout=False, with_bn=False):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16) if with_bn else nn.Identity()
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32) if with_bn else nn.Identity()
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout(0.4) if with_dropout else nn.Identity()
        self.fc = nn.Linear(32*16*16, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.flatten(1)
        x = self.drop(x)
        return self.fc(x)

@torch.no_grad()
def predict_all(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        pred = model(x).argmax(1).cpu().numpy()
        ys.append(y.numpy())
        ps.append(pred)
    return np.concatenate(ys), np.concatenate(ps)

def train_epochs(model: nn.Module, train_loader: DataLoader, dev_loader: DataLoader, epochs=4, lr=1e-3) -> Dict[str, List[float]]:
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    hist = {"train_loss": [], "dev_acc": []}
    for e in range(epochs):
        model.train()
        tot_loss, tot_n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
            tot_loss += float(loss.item()) * int(y.numel())
            tot_n += int(y.numel())
        y_true, y_pred = predict_all(model, dev_loader)
        hist["train_loss"].append(tot_loss / max(1, tot_n))
        hist["dev_acc"].append(accuracy_score(y_true, y_pred))
    return hist


## Задания (100)

- **Task 1 (10 pts)** — sanity checks для batch (`sanity_check_batch`)  
- **Task 2 (15 pts)** — метрики accuracy + macro-F1 + confusion matrix (`compute_metrics`)  
- **Task 3 (20 pts)** — LR sweep на dev и выбор лучшего LR (`lr_sweep_choose`)  
- **Task 4 (15 pts)** — дисбаланс: показать, что accuracy может быть высокой при плохом macro-F1 (`imbalance_demo`)  
- **Task 5 (20 pts)** — leakage: dev==train даёт завышенную метрику (`leakage_demo`)  
- **Task 6 (20 pts)** — `model.eval()` ошибка: метрика плавает в train-mode (`eval_mode_demo`)  


## Task 1 (10 pts) — Sanity check batch

Реализуйте `sanity_check_batch(x, y)`:
- проверяет, что `x` имеет shape `(B,3,64,64)`
- `x` float, значения в диапазоне [0,1]
- `y` int64 и значения только 0/1

Возвращает `True`, если всё ок, иначе возбуждает AssertionError.


In [ ]:
# TODO: implement
def sanity_check_batch(x: torch.Tensor, y: torch.Tensor) -> bool:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 1
try:
    tl, _ = make_loaders()
    x, y = next(iter(tl))
    ok = sanity_check_batch(x, y)
    _assert(ok is True, "sanity_check_batch must return True")
    _set_score("task1", 10)
    print("✅ Task 1 passed")
except Exception as e:
    print("❌ Task 1 failed:", e)
    _set_score("task1", 0)


## Task 2 (15 pts) — Метрики и confusion matrix

Реализуйте `compute_metrics(y_true, y_pred)`:
- `accuracy`
- `macro_f1` (average='macro')
- `cm` (confusion matrix 2x2)

Верните словарь: `{"accuracy":..., "macro_f1":..., "cm":...}`


In [ ]:
# TODO: implement
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, object]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 2
try:
    y_true = np.array([0,0,0,1,1,1])
    y_pred = np.array([0,1,0,1,0,1])
    m = compute_metrics(y_true, y_pred)
    _assert("accuracy" in m and "macro_f1" in m and "cm" in m, "Missing keys")
    _assert(m["cm"].shape == (2,2), "cm must be 2x2")
    _assert(abs(float(m["accuracy"]) - accuracy_score(y_true, y_pred)) < 1e-9, "accuracy mismatch")
    _assert(abs(float(m["macro_f1"]) - f1_score(y_true, y_pred, average='macro')) < 1e-9, "macro_f1 mismatch")
    _set_score("task2", 15)
    print("✅ Task 2 passed")
except Exception as e:
    print("❌ Task 2 failed:", e)
    _set_score("task2", 0)


## Task 3 (20 pts) — LR sweep и выбор LR

Реализуйте `lr_sweep_choose(lrs)`:
- для каждого lr обучите модель 2 эпохи на фиксированном train/dev
- верните `(best_lr, table)`:
  - `best_lr` — lr с максимальным dev accuracy (при равенстве берите меньший lr)
  - `table` — список строк: `(lr, last_train_loss, last_dev_acc)`

Используйте seed `SEED` и фиксированные loaders.


In [ ]:
# TODO: implement
def lr_sweep_choose(lrs: List[float]) -> Tuple[float, List[Tuple[float, float, float]]]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 3
try:
    lrs = [1e-5, 1e-4, 1e-3, 1e-2]
    best_lr, table = lr_sweep_choose(lrs)
    _assert(best_lr in lrs, "best_lr must be one of provided lrs")
    _assert(isinstance(table, list) and len(table) == len(lrs), "table must have one row per lr")
    # sanity: best_lr shouldn't be the extremes most of the time
    _assert(best_lr in [1e-4, 1e-3], "Expected best_lr typically in {1e-4, 1e-3} for this setup")
    _set_score("task3", 20)
    print("✅ Task 3 passed, best_lr =", best_lr)
except Exception as e:
    print("❌ Task 3 failed:", e)
    _set_score("task3", 0)


## Task 4 (15 pts) — Дисбаланс: accuracy врёт

Реализуйте `imbalance_demo()`:
- обучите модель на данных с `p_pos=0.05`
- оцените на dev и верните словарь метрик из `compute_metrics`
- условие: accuracy должна быть >= 0.90, а macro_f1 <= 0.75 (пример «обманчивой» accuracy)

Подсказка: 4-5 эпох хватает.


In [ ]:
# TODO: implement
def imbalance_demo() -> Dict[str, object]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 4
try:
    m = imbalance_demo()
    acc = float(m["accuracy"]); mf1 = float(m["macro_f1"])
    _assert(acc >= 0.90, "Expected high accuracy under imbalance")
    _assert(mf1 <= 0.75, "Expected macro_f1 to be much lower than accuracy under imbalance")
    _set_score("task4", 15)
    print("✅ Task 4 passed, acc/macro_f1 =", acc, mf1)
except Exception as e:
    print("❌ Task 4 failed:", e)
    _set_score("task4", 0)


## Task 5 (20 pts) — Leakage demo

Реализуйте `leakage_demo()`:
- обучите модель на train
- сделайте 2 оценки:
  1) честный dev (новый seed)
  2) leaky dev = train (dev==train)
- верните словарь:
  - `honest_acc`
  - `leaky_acc`
- условие: `leaky_acc - honest_acc >= 0.08`

Это демонстрация, почему leakage делает метрики «слишком красивыми».


In [ ]:
# TODO: implement
def leakage_demo() -> Dict[str, float]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 5
try:
    r = leakage_demo()
    _assert("honest_acc" in r and "leaky_acc" in r, "Missing keys")
    honest = float(r["honest_acc"]); leaky = float(r["leaky_acc"])
    _assert(leaky - honest >= 0.08, "Expected leaky_acc to be significantly higher than honest_acc")
    _set_score("task5", 20)
    print("✅ Task 5 passed, honest/leaky =", honest, leaky)
except Exception as e:
    print("❌ Task 5 failed:", e)
    _set_score("task5", 0)


## Task 6 (20 pts) — Ошибка `model.eval()` (Dropout/BatchNorm)

Реализуйте `eval_mode_demo()`:
- обучите модель `SmallCNN(with_dropout=True, with_bn=True)` на честных данных
- посчитайте:
  - `acc_eval` — accuracy при правильной оценке (model.eval)
  - `acc_train_mode` — список из 5 значений accuracy, если **оценивать в train-режиме**
- верните словарь:
  - `acc_eval`
  - `acc_train_mode` (list[float])
  - `spread` = max - min по acc_train_mode
- условие: spread >= 0.02 (метрика реально «плавает»)

Подсказка: 3-4 эпох хватает.


In [ ]:
# TODO: implement
def eval_mode_demo() -> Dict[str, object]:
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Tests Task 6
try:
    r = eval_mode_demo()
    _assert("acc_eval" in r and "acc_train_mode" in r and "spread" in r, "Missing keys")
    acc_eval = float(r["acc_eval"])
    acc_list = r["acc_train_mode"]
    spread = float(r["spread"])
    _assert(isinstance(acc_list, list) and len(acc_list) == 5, "acc_train_mode must be list of 5 floats")
    _assert(spread >= 0.02, "Expected metric spread >= 0.02 when evaluating in train-mode")
    _set_score("task6", 20)
    print("✅ Task 6 passed, acc_eval/spread =", acc_eval, spread)
except Exception as e:
    print("❌ Task 6 failed:", e)
    _set_score("task6", 0)


In [ ]:
# FINAL: compute total points (0..100)
total = _total_score()
_print_scores()
print("\nTOTAL POINTS / 100 =", total)


In [ ]:
def penalty_fraction(start_dt, due_dt, now_dt) -> float:
    if not (start_dt and due_dt and now_dt):
        return 0.0
    window = _sec(due_dt - start_dt)
    if window <= 0:
        return 1.0 if now_dt > due_dt else 0.0
    late = max(0.0, _sec(now_dt - due_dt))
    return min(1.0, late / window)
	
# применяем штраф
try:
    pf = penalty_fraction(start_dt, due_dt, submission_dt)
except NameError:
    from datetime import timezone
    pf = 0.0
final_score = max(0.0, total * (1.0 - min(1.0, pf)))
print(final_score)
import json
final = {
    "name": STUDENT_NAME,
    "group": GROUP,
    "assignment":ASSIGMENT,
    "score": float(total),
    "penalty_score": float(final_score),
    "due_date": DUE_DATE,
    "start_date": START_DATE,
}
print(json.dumps(final, ensure_ascii=False))
